# 🎯 CS2 Bot Training - Обучение на конкретном игроке
## Копируем стиль про-игрока через FACEIT API

**Что делает:**
1. Скачивает демки конкретного игрока с FACEIT
2. Парсит ТОЛЬКО его действия (движение, aim, гранаты, тактика)
3. Обучает AI копировать его стиль игры

**Требования:**
- GPU Runtime (Runtime → Change runtime type → T4 GPU)
- FACEIT API ключ (получи на https://developers.faceit.com/)

**Популярные игроки:** ZywOo, donk, s1mple, NiKo, m0NESY, electronic

## 🔧 Шаг 1: Проверка GPU

In [ ]:
# Проверка GPU (должен быть Tesla T4 или другой)
!nvidia-smi

## 📦 Шаг 2: Установка библиотек

In [ ]:
# Установка всех необходимых библиотек (~2 минуты)
!pip install -q requests awpy pandas pyarrow tqdm torch

print("✅ Библиотеки установлены!")

## 📥 Шаг 3: Клонирование репозитория

In [ ]:
# Удаляем старые копии если есть
!rm -rf /content/cs2-bot-training

# Переходим в корень
%cd /content

# Клонируем репозиторий
!git clone https://github.com/nem1k9/cs2-bot-training.git

# Переходим в папку scripts (там все .py файлы)
%cd cs2-bot-training/scripts

# Проверяем что файлы есть
print("\n✅ Файлы проекта:")
!ls -la

## 🎮 Шаг 4: Скачивание демок игрока с FACEIT

**ВАЖНО:**
1. Получи API ключ на https://developers.faceit.com/
2. Вставь его в `API_KEY` ниже
3. Выбери игрока в `PLAYER`

**Популярные игроки:**
- AWP: `ZywOo`, `m0NESY`, `sh1ro`
- Rifle: `donk`, `NiKo`, `electronic`
- Entry: `rain`, `YEKINDAR`

In [ ]:
from download_faceit_demos import download_player_demos_simple

# ========== НАСТРОЙКИ ==========
API_KEY = "YOUR_API_KEY_HERE"  # ← ВСТАВЬ СВОЙ API КЛЮЧ СЮДА!
PLAYER = "ZywOo"  # ← Выбери игрока: ZywOo, donk, s1mple, NiKo и т.д.
N_DEMOS = 10  # Количество демок (10 = ~5-8 GB, 20 = ~10-16 GB)
# ===============================

print(f"🎯 Скачиваем {N_DEMOS} демок игрока: {PLAYER}")
print(f"⏱️  Это займёт ~10-20 минут...\n")

download_player_demos_simple(
    nickname=PLAYER,
    api_key=API_KEY,
    n_demos=N_DEMOS,
    save_dir="./demos"
)

print(f"\n✅ Демки игрока {PLAYER} скачаны!")

## 🔍 Шаг 5: Парсинг с фокусом на игрока

**Что отслеживается:**
- Движение (позиция, скорость, присед)
- Прицеливание (yaw, pitch, скоп)
- Стрельба (выстрелы, перезарядка)
- Гранаты (флешки, смоки, HE, молотовы)
- Тактика (позиционирование, дистанция до врагов)

In [ ]:
from parse_demos import build_dataset

print(f"🔄 Парсим демки для игрока: {PLAYER}")
print(f"⏱️  Это займёт ~10-30 минут...\n")

# Парсим ТОЛЬКО действия выбранного игрока
dataset = build_dataset(
    demo_dir="./demos",
    out_path="./dataset.parquet",
    target_player=PLAYER  # ← Фильтр по игроку
)

print(f"\n✅ Датасет для {PLAYER} готов: {len(dataset):,} тиков")
print(f"📊 Размер датасета: {dataset.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 🚀 Шаг 6: Обучение модели

**Время обучения:** ~2-4 часа

**ВАЖНО:** Не закрывай вкладку браузера!

In [ ]:
from train import train

print(f"🚀 Начинаем обучение модели на стиле {PLAYER}...")
print(f"⏱️  Это займёт ~2-4 часа...\n")

train()

print(f"\n✅ Обучение завершено!")
print(f"🎯 Модель обучена копировать стиль игры {PLAYER}!")

## 💾 Шаг 7: Сохранение результатов на Google Drive

In [ ]:
# Подключаем Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Создаём папку для результатов
!mkdir -p /content/drive/MyDrive/cs2_bot_results

# Сохраняем модели и датасет
print("💾 Сохраняем результаты на Google Drive...\n")

!cp cs2bot_final.pt /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ cs2bot_final.pt сохранён" || echo "⚠️  cs2bot_final.pt не найден"
!cp checkpoint.pt /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ checkpoint.pt сохранён" || echo "⚠️  checkpoint.pt не найден"
!cp dataset.parquet /content/drive/MyDrive/cs2_bot_results/ 2>/dev/null && echo "✅ dataset.parquet сохранён" || echo "⚠️  dataset.parquet не найден"

print("\n✅ Результаты сохранены на Google Drive в папке cs2_bot_results!")
print("\n📁 Что сохранено:")
!ls -lh /content/drive/MyDrive/cs2_bot_results/

## 📥 Шаг 8: Скачать модель на компьютер (опционально)

In [ ]:
# Скачать модель прямо в браузер
from google.colab import files

print("📥 Скачиваем модель...")
files.download('cs2bot_final.pt')
files.download('checkpoint.pt')

print("✅ Модель скачана!")

## 🎯 Готово!

**Что получилось:**
- ✅ Скачаны демки игрока {PLAYER}
- ✅ Распарсены все его действия
- ✅ Обучена модель копирующая его стиль
- ✅ Результаты сохранены на Google Drive

**Модель умеет:**
- Двигаться как {PLAYER}
- Целиться как он
- Использовать гранаты в похожих ситуациях
- Принимать тактические решения как про-игрок

**Это AI-копия стиля игры {PLAYER}!** 🔥